In [ ]:
# Cell 1: Dependencies
#!pip install google-generativeai datasets pandas pyarrow huggingface_hub langid

# Cell 2: Imports
import pandas as pd
import numpy as np
from datasets import Dataset, load_dataset, DatasetDict
import os
import json
import time
import google.generativeai as genai
from datetime import datetime
import re
import langid
from collections import Counter
import hashlib
import random
from huggingface_hub import login, upload_file, hf_hub_download

In [ ]:
# Cell 3: Enhanced Legal Corpus Generation Configuration
DOMAIN_CONFIG = {
    'domain': 'Comprehensive Legal Corpus',
    'target_language': 'en',  # or 'id' for Indonesian
    'generation_mode': 'full_document',
    'include_sentiment_analysis': True,
}

# Legal corpus taxonomy - maps to your synthetic_legal_corpus dictionary
LEGAL_CORPUS_TAXONOMY = {
    'corpus_categories': [
        "Corporate & Commercial",
        "Contracts & Negotiation",
        "Litigation & Dispute",
        "Legal Analysis & Advisory",
        "Regulatory & Compliance",
        "Employment & HR",
        "Real Estate & Property",
        "Finance, Banking & Insurance",
        "International & Trade",
        "Emerging Tech & IP",
        "Civil & Criminal Procedure",
        "Legal Education & Training",
        "Meta-Legal Reasoning & Strategic Intelligence"
    ],
    
    'document_types': {
        "Corporate & Commercial": [
            "Company Policies and Code of Conduct",
            "Corporate Governance Manuals",
            "Compliance Handbooks and Internal Guidelines",
            "Risk Disclosure Statements",
            "M&A Due Diligence Reports",
            "Transaction Term Sheets",
            "Board Meeting Minutes and Resolutions",
            "Internal Audit Reports",
            "Procurement Policy Documents",
            "Vendor Evaluation Forms",
        ],
        "Contracts & Negotiation": [
            "Clause Libraries by Legal Topic",
            "Contract Addendums and Side Letters",
            "Redlined Contract Versions",
            "Negotiation Transcripts between Counsel",
            "Lawyer–Client Advisory Memos",
            "Contract Risk Annotations and Commentaries",
            "Clause Variant Libraries",
            "Confidential Side Agreements",
            "Term Sheets and Deal Summaries",
            "Internal Draft Review Notes",
        ],
        "Litigation & Dispute": [
            "Pleadings and Complaints",
            "Answers, Replies, and Counterclaims",
            "Motions and Petitions",
            "Court Orders (Synthetic Drafts)",
            "Judgment and Verdict Summaries",
            "Settlement Agreements",
            "Legal Briefs and Memoranda of Law",
            "Evidence Summaries and Descriptions",
            "Witness Statements and Depositions",
            "Full Case File Compilations",
        ],
        "Legal Analysis & Advisory": [
            "Internal Legal Memos and Notes",
            "Formal Legal Opinions and Advisory Letters",
            "Case Analysis and Reasoning Summaries",
            "Comparative Law Reports",
            "Risk Assessment Reports and Matrices",
            "Interpretation Guides and Policy Explanations",
            "Regulatory Impact Assessments",
            "Legislative Commentary Drafts",
            "Internal Legal Research Notes",
            "Counsel Review Summaries",
        ],
        "Regulatory & Compliance": [
            "Compliance Checklists and Evaluation Sheets",
            "Regulatory Filing Templates and Forms",
            "Self-Audit Reports and Control Logs",
            "Inspection Reports",
            "Policy Implementation Guides",
            "Data Protection Impact Assessments",
            "Licensing Application Examples",
            "Regulatory Correspondence Letters",
            "Internal Compliance Monitoring Reports",
            "Industry-Specific Compliance Manuals",
        ],
        "Employment & HR": [
            "Employee Handbooks and Policy Guides",
            "HR Compliance Manuals",
            "Disciplinary Procedure Templates",
            "Whistleblower Protection Policies",
            "Labor Dispute Case Summaries",
            "Internal Complaint Reports",
            "Workplace Investigation Reports",
            "Employment Offer Letters",
            "Termination and Severance Agreements",
            "Performance Contract Templates",
        ],
        "Real Estate & Property": [
            "Lease Templates",
            "Title Transfer Forms and Declarations",
            "Property Sale and Purchase Agreements",
            "Mortgage and Financing Contracts",
            "Zoning and Planning Compliance Summaries",
            "Property Due Diligence Reports",
            "Environmental Compliance Documents",
            "Construction Contract Variants",
            "Inspection Certificates and Reports",
            "Landlord–Tenant Dispute Files",
        ],
        "Finance, Banking & Insurance": [
            "Loan Agreement Variants",
            "Credit Risk Evaluation Reports",
            "KYC and AML Policy Documents",
            "Investment Prospectus Sections",
            "Insurance Policy Templates",
            "Claim Processing Reports",
            "Financial Regulatory Filings (Synthetic)",
            "Underwriting Guidelines",
            "Audit Reports (Synthetic Narratives)",
            "Risk Disclosure Summaries",
        ],
        "International & Trade": [
            "Trade Agreement Drafts",
            "Customs Compliance Manuals",
            "Import–Export Policy Guides",
            "Sanctions Screening Procedures",
            "Cross-Border Transaction Summaries",
            "Treaty Interpretation Notes",
            "Diplomatic Communication Samples",
            "International Arbitration Clauses",
            "Supply Chain Compliance Documents",
            "Global Regulatory Framework Comparisons",
        ],
        "Emerging Tech & IP": [
            "Data Privacy Policy Templates",
            "Cybersecurity Incident Reports",
            "AI Governance and Ethics Guidelines",
            "Technology Licensing Agreements",
            "Patent Assignment Declarations",
            "Digital Evidence Reports",
            "Terms of Use and Privacy Policy Variants",
            "Blockchain Smart Contract Summaries",
            "AI Bias and Ethics Audit Reports",
            "Software Compliance Declarations",
        ],
        "Civil & Criminal Procedure": [
            "Police Reports (Synthetic Narratives)",
            "Investigation Summaries",
            "Civil Case Summaries",
            "Sentencing Memoranda",
            "Plea Bargain Templates",
            "Evidence Chain-of-Custody Forms",
            "Expert Witness Reports",
            "Jury Instruction Examples",
            "Victim or Witness Statements",
            "Appeal Briefs (Synthetic Cases)",
        ],
        "Legal Education & Training": [
            "Hypothetical Case Studies",
            "Law School Exam Problems",
            "Legal Essay Examples",
            "Moot Court Case Packs",
            "Legal Drafting Exercises",
            "Model Case Answers and Summaries",
            "Reasoning Practice Scenarios",
            "Legal Ethics and Professional Conduct Scenarios",
            "Debate or Oral Argument Transcripts",
            "Synthetic Academic Law Articles",
        ],
        "Meta-Legal Reasoning & Strategic Intelligence": [
            "Negotiation Psychology Notes",
            "Power Dynamics and Leverage Analysis",
            "Strategic Litigation Planning Documents",
            "Judge Behavioral Profiles",
            "Jury Sentiment Prediction Models",
            "Evidentiary Strength Assessment Memos",
            "Moral vs. Tactical Decision Case Studies",
            "Risk Appetite and Exposure Evaluation Reports",
            "Synthetic Crisis Response Playbooks",
            "Cross-Border Regulatory Risk Assessment Reports",
        ]
    },
    
    'industries': [
        "Technology & Software",
        "Finance & Banking",
        "Healthcare & Pharmaceuticals",
        "Manufacturing & Industrial",
        "Energy & Utilities",
        "Real Estate & Construction",
        "Government & Public Sector",
        "Retail & E-commerce",
        "Transportation & Logistics",
        "Telecommunications",
        "Education & Research",
        "Media & Entertainment"
    ],
    
    'jurisdictions': [
        "US Federal Law",
        "European Union Law",
        "Singapore Law",
        "Indonesian Law",
        "UK Law",
        "International Law",
        "Australian Law",
        "Canadian Law",
    ],
    
    'perspectives': [
        "Plaintiff/Claimant",
        "Defendant/Respondent",
        "Neutral Observer",
        "Regulatory Authority",
        "Corporate Counsel",
        "External Advisor",
        "Investigator",
        "Compliance Officer"
    ],
    
    'sentiment_bias': [
        "Strongly Favorable",  # +2
        "Moderately Favorable",  # +1
        "Neutral/Balanced",  # 0
        "Moderately Critical",  # -1
        "Strongly Critical"  # -2
    ],
    
    'complexity_levels': [
        "Basic/Introductory",
        "Intermediate",
        "Advanced",
        "Expert/Technical"
    ],
    
    'metadata': {
        'document_length': ["Short (1-3 pages)", "Medium (3-8 pages)", "Long (8-15 pages)", "Very Long (15+ pages)"],
        'formality': ["Formal", "Semi-Formal", "Technical"],
        'urgency': ["Routine", "Important", "Urgent", "Critical"],
    }
}

# Generation prompts
CORPUS_GENERATION_PROMPT = """Generate a complete, realistic {document_type} in the context of {category}.

DOCUMENT CONTEXT:
- Category: {category}
- Industry/Sector: {industry}
- Jurisdiction: {jurisdiction}
- Perspective: {perspective}
- Complexity Level: {complexity}
- Target Length: {length}
- Sentiment/Bias: {sentiment}

REQUIRED CHARACTERISTICS:
1. Authentic legal language appropriate for {jurisdiction}
2. Perspective should reflect {perspective} viewpoint with {sentiment} tone
3. Include realistic specifics: dates, amounts, parties, citations, references
4. Appropriate technical depth for {complexity} level
5. Natural formatting and structure for this document type
6. If sentiment is non-neutral, incorporate subtle bias through:
   - Word choice and framing
   - Emphasis on certain facts over others
   - Tone and interpretation
   - Risk characterization

SENTIMENT GUIDANCE:
- Strongly Favorable: Emphasize positives, minimize concerns, optimistic framing
- Moderately Favorable: Generally positive but acknowledge some limitations
- Neutral/Balanced: Objective, even-handed presentation
- Moderately Critical: Point out concerns, cautious tone, identify risks
- Strongly Critical: Emphasize problems, skeptical framing, highlight dangers

The document should feel authentic with:
- Real-world language and terminology
- Industry-appropriate references
- Realistic scenarios and fact patterns
- Natural variation in structure
- Actual details (not placeholders)

Generate the complete document now. Match the complexity and length requirements.

DOCUMENT:"""

SENTIMENT_ANALYSIS_PROMPT = """Analyze the sentiment, bias, and characteristics of this legal document:

{document_preview}

Context: {document_type} in {category}
Perspective: {perspective}

Provide a structured analysis (300-400 words):

1. SENTIMENT SCORE (-2 to +2):
   - Overall sentiment rating
   - Key indicators of bias
   - Tone characterization

2. PERSPECTIVE ANALYSIS:
   - How perspective influences presentation
   - Whose interests are prioritized
   - Balanced vs. advocate analysis

3. RISK CHARACTERIZATION:
   - How risks are framed
   - Severity language
   - Mitigation emphasis

4. CONTENT QUALITY:
   - Technical accuracy level
   - Completeness
   - Professional standard

ANALYSIS:"""

SYSTEM_PROMPT = """You are an experienced legal professional with expertise across multiple practice areas and jurisdictions.

You create authentic legal documents that reflect real-world scenarios, including documents that:
- Represent different perspectives (plaintiff, defendant, neutral, regulatory)
- Contain appropriate sentiment and bias based on the author's position
- Use jurisdiction-appropriate language and citations
- Match the complexity level required
- Feel genuine, not templated

You understand that legal documents are rarely purely neutral - they often:
- Advocate for a position
- Frame facts strategically
- Emphasize certain aspects over others
- Reflect the author's perspective and interests

Your documents are realistic, professionally written, and contextually appropriate."""

In [ ]:
# Cell 4: Main Configuration
CONFIG = {
    #'gemini_api_key': '',
    'gemini_api_key': '',
    #'gemini_api_key': '',
    #'gemini_api_key': '',
    #'gemini_api_key': '',
    
    #'gemini_api_key': '',
    #'gemini_api_key': '',
    #'gemini_api_key': '',
    #'gemini_api_key': '',
    #'gemini_api_key': '',
    
    #'gemini_api_key': '',
    #'gemini_api_key': '',
    
    'huggingface_token': '',
    'output_repository': 'Azzindani/Legal_Corpus_Syn',
    
    # Optimized settings
    'batch_size': 5,  # Documents per batch
    'model_name': 'gemini-2.5-flash',
    'temperature': 0.85,
    
    # Token optimization
    'max_output_tokens': 50000,
    'enable_sentiment_analysis': False,  # Set True for detailed sentiment analysis (costs extra tokens)
    
    # Quality settings
    'min_contract_length': 5000,  # Characters minimum (changed from min_contract_length to be consistent)
    'target_language_confidence': 0.5,
    
    # Processing control
    'skip_processed_rows': True,
    'progress_file': 'synthesis_progress.json',
    
    # Safety settings (allow legal terminology)
    'safety_settings': {
        'HARM_CATEGORY_HARASSMENT': 'BLOCK_NONE',
        'HARM_CATEGORY_HATE_SPEECH': 'BLOCK_NONE',
        'HARM_CATEGORY_SEXUALLY_EXPLICIT': 'BLOCK_NONE',
        'HARM_CATEGORY_DANGEROUS_CONTENT': 'BLOCK_ONLY_HIGH',
    }
}

In [ ]:
# Cell 5: Authentication
genai.configure(api_key=CONFIG['gemini_api_key'])
login(token=CONFIG['huggingface_token'])

In [ ]:
# Cell 6: Text Validator for Single-Shot Processing
class TextValidator:
    def __init__(self):
        self.max_length = DOMAIN_CONFIG['max_text_length']
        self.min_length = DOMAIN_CONFIG['min_text_length']
    
    def validate_and_prepare_text(self, text):
        """Validate and prepare text for single-shot processing"""
        if not text or not isinstance(text, str):
            return None
        
        text = text.strip()
        
        if len(text) < self.min_length:
            return None
        
        # Truncate if too long (preserve beginning and end)
        if len(text) > self.max_length:
            half_max = self.max_length // 2 - 100  # Leave room for separator
            text = text[:half_max] + "\n\n[...TRUNCATED...]\n\n" + text[-half_max:]
        
        return text

In [ ]:
# Cell 7: FIXED Language Detection
class LanguageScorer:
    def __init__(self, target_language='en'):
        self.target_language = target_language
    
    def detect_language(self, text):
        """Detect language - FIX for langid's negative confidence"""
        if not text or len(text.strip()) < 10:
            return 'unknown', 0.0
        
        try:
            lang, raw_confidence = langid.classify(text.strip())
            
            # CRITICAL FIX: langid returns NEGATIVE log probability
            # Convert to 0-1 scale using exponential
            # More negative = less confident, closer to 0 = more confident
            import math
            
            # Normalize: convert negative log prob to probability
            # Typical range is -inf to 0, most values between -5 and -0.5
            if raw_confidence <= 0:
                # More confident (closer to 0) = higher score
                normalized_confidence = math.exp(raw_confidence)
                # This gives us 0.0 to 1.0 where 1.0 is most confident
            else:
                # Shouldn't happen, but safety check
                normalized_confidence = 1.0
            
            # Clamp to valid range
            normalized_confidence = max(0.0, min(1.0, normalized_confidence))
            
            return lang, normalized_confidence
            
        except Exception as e:
            print(f"    Language detection error: {e}")
            return 'unknown', 0.0
    
    def score_corpus_language(self, corpus_text):
        """Language detection for corpus text"""
        if not corpus_text or len(corpus_text.strip()) < 50:
            return {
                'detected_language': 'unknown',
                'average_confidence': 0.0,
                'language_consistent': False,
                'matches_target': False,
            }
        
        # Sample from different parts
        text_len = len(corpus_text)
        samples = []
        
        if text_len > 0:
            samples.append(corpus_text[:min(500, text_len)])
        if text_len > 1000:
            mid = text_len // 2
            samples.append(corpus_text[mid:min(mid+500, text_len)])
        if text_len > 500:
            samples.append(corpus_text[-500:])
        
        # Detect language for each sample
        detections = []
        for sample in samples:
            if len(sample.strip()) < 10:
                continue
            lang, conf = self.detect_language(sample)
            if lang != 'unknown' and conf > 0:
                detections.append((lang, conf))
                print(f"    Sample detected: {lang} (confidence: {conf:.3f})")
        
        if not detections:
            return {
                'detected_language': 'unknown',
                'average_confidence': 0.0,
                'language_consistent': False,
                'matches_target': False,
            }
        
        # Calculate statistics
        languages = [d[0] for d in detections]
        confidences = [d[1] for d in detections]
        
        most_common_lang = max(set(languages), key=languages.count)
        avg_confidence = sum(confidences) / len(confidences)
        is_consistent = all(lang == most_common_lang for lang in languages)
        
        print(f"    Final language: {most_common_lang}, avg confidence: {avg_confidence:.3f}")
        
        return {
            'detected_language': str(most_common_lang),
            'average_confidence': float(avg_confidence),
            'language_consistent': bool(is_consistent),
            'matches_target': bool(most_common_lang == self.target_language),
        }

In [ ]:
# Cell 8: Progress Manager (adapted for corpus)
class ProgressManager:
    def __init__(self):
        self.progress_data = {
            'processed_chunks': [],
            'current_row': 0,
            'total_chunks_processed': 0,
            'total_qa_pairs_created': 0,
            'request_count': 0,
            'start_time': None,
            'last_update': None,
            'errors': [],
            'statistics': {
                'avg_chunks_per_text': 0.0,
                'avg_qa_per_chunk': 0.0,
                'approach_stats': {'simple': 0, 'deep_thinking': 0, 'iterative_thinking': 0}
            }
        }
        self.load_progress()
    
    def load_progress(self):
        try:
            progress_path = hf_hub_download(
                repo_id=CONFIG['output_repository'],
                filename=CONFIG['progress_file'],
                repo_type="dataset"
            )
            with open(progress_path, 'r') as f:
                saved_progress = json.load(f)
                self.progress_data.update(saved_progress)
            print(f"Progress loaded: {self.progress_data['total_chunks_processed']} chunks processed")
        except Exception as e:
            print(f"No existing progress found, starting fresh: {e}")
            self.progress_data['start_time'] = datetime.now().isoformat()
    
    def save_progress(self):
        try:
            self.progress_data['last_update'] = datetime.now().isoformat()
            
            def convert_types(obj):
                if isinstance(obj, dict):
                    return {k: convert_types(v) for k, v in obj.items()}
                elif isinstance(obj, list):
                    return [convert_types(v) for v in obj]
                elif hasattr(obj, 'item'):
                    return obj.item()
                elif hasattr(obj, 'tolist'):
                    return obj.tolist()
                else:
                    return obj
            
            clean_data = convert_types(self.progress_data)
            local_path = f"./{CONFIG['progress_file']}"
            
            with open(local_path, 'w') as f:
                json.dump(clean_data, f, indent=2)
            
            upload_file(
                path_or_fileobj=local_path,
                path_in_repo=CONFIG['progress_file'],
                repo_id=CONFIG['output_repository'],
                repo_type="dataset",
                commit_message=f"Corpus progress: {self.progress_data['total_qa_pairs_created']} QA pairs"
            )
            print(f"Progress saved to repository")
        except Exception as e:
            print(f"Failed to save progress: {e}")
    
    def is_chunk_processed(self, chunk_id):
        return chunk_id in self.progress_data['processed_chunks']
    
    def mark_chunk_processed(self, chunk_id, qa_count):
        if chunk_id not in self.progress_data['processed_chunks']:
            self.progress_data['processed_chunks'].append(chunk_id)
            self.progress_data['total_chunks_processed'] += 1
            self.progress_data['total_qa_pairs_created'] += qa_count

In [ ]:
# Cell 9: Enhanced Legal Corpus Generator
class LegalCorpusGenerator:
    def __init__(self, progress_manager):
        self.model = genai.GenerativeModel(
            CONFIG['model_name'],
            generation_config=genai.types.GenerationConfig(
                temperature=CONFIG['temperature'],
                max_output_tokens=CONFIG['max_output_tokens'],
                top_p=0.9,
                top_k=40
            ),
            system_instruction=SYSTEM_PROMPT,
            safety_settings=[
                {"category": "HARM_CATEGORY_HARASSMENT", "threshold": "BLOCK_NONE"},
                {"category": "HARM_CATEGORY_HATE_SPEECH", "threshold": "BLOCK_NONE"},
                {"category": "HARM_CATEGORY_SEXUALLY_EXPLICIT", "threshold": "BLOCK_NONE"},
                {"category": "HARM_CATEGORY_DANGEROUS_CONTENT", "threshold": "BLOCK_ONLY_HIGH"}
            ]
        )
        
        self.language_scorer = LanguageScorer(DOMAIN_CONFIG['target_language'])
        self.progress_manager = progress_manager
        self.request_count = progress_manager.progress_data['request_count']
    
    def generate_scenario_combination(self):
        """Generate random legal document scenario"""
        category = random.choice(LEGAL_CORPUS_TAXONOMY['corpus_categories'])
        document_types = LEGAL_CORPUS_TAXONOMY['document_types'][category]
        
        scenario = {
            'category': category,
            'document_type': random.choice(document_types),
            'industry': random.choice(LEGAL_CORPUS_TAXONOMY['industries']),
            'jurisdiction': random.choice(LEGAL_CORPUS_TAXONOMY['jurisdictions']),
            'perspective': random.choice(LEGAL_CORPUS_TAXONOMY['perspectives']),
            'sentiment': random.choice(LEGAL_CORPUS_TAXONOMY['sentiment_bias']),
            'complexity': random.choice(LEGAL_CORPUS_TAXONOMY['complexity_levels']),
            'length': random.choice(LEGAL_CORPUS_TAXONOMY['metadata']['document_length']),
            'formality': random.choice(LEGAL_CORPUS_TAXONOMY['metadata']['formality']),
        }
        return scenario
    
    def sentiment_to_score(self, sentiment_text):
        """Convert sentiment text to numeric score"""
        mapping = {
            "Strongly Favorable": 2,
            "Moderately Favorable": 1,
            "Neutral/Balanced": 0,
            "Moderately Critical": -1,
            "Strongly Critical": -2
        }
        return mapping.get(sentiment_text, 0)
    
    def generate_document(self, scenario, corpus_id):
        """Generate realistic legal document"""
        prompt = CORPUS_GENERATION_PROMPT.format(**scenario)
        
        try:
            time.sleep(6)
            response = self.model.generate_content(prompt)
            self.request_count += 1
            self.progress_manager.progress_data['request_count'] = self.request_count
            
            if not response.candidates:
                print(f"    Document {corpus_id} - No response")
                return None, None
            
            candidate = response.candidates[0]
            finish_reason = candidate.finish_reason
            
            if finish_reason == 2:  # Safety filtered
                print(f"    Document {corpus_id} - Safety filtered, retrying with safer approach")
                return self.generate_safe_fallback(scenario, corpus_id)
            
            document_text = candidate.content.parts[0].text.strip()
            
            if finish_reason == 3:  # Max tokens
                print(f"    Document {corpus_id} - Reached max tokens ({len(document_text)} chars)")
            
            print(f"    Generated: {len(document_text)} chars, {len(document_text.split())} words")
            
            if len(document_text) < CONFIG['min_contract_length']:
                print(f"    Document too short, skipping")
                return None, None
            
            sentiment_data = self.analyze_document_sentiment(document_text, scenario)
            
            return document_text, sentiment_data
            
        except Exception as e:
            print(f"    Error: {e}")
            return None, None
    
    def generate_safe_fallback(self, scenario, corpus_id):
        """Fallback with extra-safe language when filtered"""
        safe_prompt = f"""Draft a professional {scenario['document_type']} for the {scenario['industry']} industry under {scenario['jurisdiction']}.

Context:
- Document category: {scenario['category']}
- Complexity level: {scenario['complexity']}
- Perspective: {scenario['perspective']}

Create a complete, professional document with:
- Appropriate structure and formatting
- Realistic content and terminology
- Professional tone
- Contextually appropriate detail

Use standard professional language.

DOCUMENT:"""
        
        try:
            time.sleep(6)
            response = self.model.generate_content(safe_prompt)
            self.request_count += 1
            
            if not response.candidates or response.candidates[0].finish_reason == 2:
                print(f"    Fallback also filtered - skipping")
                return None, None
            
            document_text = response.candidates[0].content.parts[0].text.strip()
            
            if len(document_text) < CONFIG['min_contract_length']:
                return None, None
            
            sentiment_data = self.analyze_document_sentiment(document_text, scenario)
            return document_text, sentiment_data
            
        except Exception as e:
            print(f"    Fallback failed: {e}")
            return None, None
    
    def analyze_document_sentiment(self, document_text, scenario):
        """Analyze document sentiment and characteristics"""
        
        word_count = len(document_text.split())
        
        # Basic heuristic scoring
        intended_score = self.sentiment_to_score(scenario['sentiment'])
        
        # Optional: Detailed analysis (costs extra API call)
        if CONFIG.get('enable_sentiment_analysis', False):
            detailed_analysis = self.get_detailed_sentiment(document_text[:2000], scenario)
        else:
            detailed_analysis = f"{scenario['document_type']} presenting {scenario['perspective']} perspective with {scenario['sentiment']} tone in {scenario['category']} context."
        
        return {
            'intended_sentiment': scenario['sentiment'],
            'sentiment_score': intended_score,
            'perspective': scenario['perspective'],
            'category': scenario['category'],
            'complexity': scenario['complexity'],
            'analysis_text': detailed_analysis
        }
    
    def get_detailed_sentiment(self, document_preview, scenario):
        """Optional detailed sentiment analysis"""
        prompt = SENTIMENT_ANALYSIS_PROMPT.format(
            document_preview=document_preview,
            document_type=scenario['document_type'],
            category=scenario['category'],
            perspective=scenario['perspective']
        )
        
        try:
            time.sleep(4)
            response = self.model.generate_content(prompt)
            self.request_count += 1
            
            if response.candidates and response.candidates[0].finish_reason not in [2, 4]:
                return response.candidates[0].content.parts[0].text.strip()
        except:
            pass
        
        return f"Analysis unavailable - standard {scenario['document_type']} with {scenario['sentiment']} perspective."
    
    def generate_batch(self, batch_size):
        """Generate batch of legal documents"""
        results = []
        
        for i in range(batch_size):
            print(f"\nGenerating document {i+1}/{batch_size}")
            
            scenario = self.generate_scenario_combination()
            print(f"  {scenario['document_type']}")
            print(f"  Category: {scenario['category']} | Industry: {scenario['industry']}")
            print(f"  Perspective: {scenario['perspective']} | Sentiment: {scenario['sentiment']}")
            
            document_text, sentiment_data = self.generate_document(scenario, i)
            
            if not document_text or not sentiment_data:
                continue
            
            language_scores = self.language_scorer.score_corpus_language(document_text)
            
            result = {
                'corpus_id': f"legal_{self.progress_manager.progress_data['total_qa_pairs_created'] + len(results):05d}",
                'document_text': document_text,
                'text_length': len(document_text),
                'word_count': len(document_text.split()),
                
                # Document metadata
                'category': scenario['category'],
                'document_type': scenario['document_type'],
                'industry': scenario['industry'],
                'jurisdiction': scenario['jurisdiction'],
                'complexity_level': scenario['complexity'],
                'target_length': scenario['length'],
                'formality': scenario['formality'],
                
                # Sentiment and perspective
                'perspective': sentiment_data['perspective'],
                'intended_sentiment': sentiment_data['intended_sentiment'],
                'sentiment_score': sentiment_data['sentiment_score'],
                'sentiment_analysis': sentiment_data['analysis_text'],
                
                # Language
                **language_scores,
                
                'timestamp': datetime.now().isoformat(),
            }
            
            results.append(result)
            print(f"  ✓ Complete: {result['word_count']} words | Sentiment: {result['sentiment_score']} ({result['intended_sentiment']})")
        
        return results

In [ ]:
# Cell 10: Utility Functions for Legal Corpus
def save_corpus_batch(results, batch_start):
    """Save corpus batch with standardized format"""
    if not results:
        return
    
    try:
        df = pd.DataFrame(results)
        
        # Standardize schema
        df = df.astype({
            'corpus_id': 'string',
            'document_text': 'string',
            'category': 'string',
            'document_type': 'string',
            'industry': 'string',
            'jurisdiction': 'string',
            'complexity_level': 'string',
            'target_length': 'string',
            'formality': 'string',
            'perspective': 'string',
            'intended_sentiment': 'string',
        })
        
        batch_id = f"{batch_start:05d}-{batch_start+len(results):05d}"
        filename = f"train-{batch_id}.parquet"
        
        temp_filepath = f"/tmp/{filename}"
        df.to_parquet(temp_filepath, index=False, engine='pyarrow')
        
        upload_file(
            path_or_fileobj=temp_filepath,
            path_in_repo=filename,
            repo_id=CONFIG['output_repository'],
            repo_type="dataset",
            commit_message=f"Legal corpus batch {batch_id}: {len(results)} documents"
        )
        
        print(f"  ✓ Uploaded {filename} ({len(results)} corpus documents)")
        
        os.remove(temp_filepath)
        
    except Exception as e:
        print(f"Failed to save corpus batch: {e}")

def process_legal_corpus_batch(target_count):
    """Generate a batch of legal corpus documents"""
    generator = LegalCorpusGenerator(progress_manager)
    
    already_generated = progress_manager.progress_data.get('total_qa_pairs_created', 0)
    remaining = target_count - already_generated
    
    if remaining <= 0:
        print(f"Target of {target_count} corpus documents already reached")
        return
    
    batch_size = min(remaining, CONFIG['batch_size'])
    
    print(f"Generating {batch_size} legal corpus documents ({already_generated}/{target_count} complete)")
    
    results = generator.generate_batch(batch_size)
    
    if results:
        save_corpus_batch(results, already_generated)
        
        progress_manager.progress_data['total_qa_pairs_created'] += len(results)
        progress_manager.save_progress()
        
        print(f"\nBatch complete: {len(results)} corpus documents generated")
        print(f"Total progress: {progress_manager.progress_data['total_qa_pairs_created']}/{target_count}")

def continue_legal_corpus_generation(target_count=100):
    """Continue generating legal corpus until target reached"""
    while progress_manager.progress_data.get('total_qa_pairs_created', 0) < target_count:
        process_legal_corpus_batch(target_count)

def show_legal_corpus_stats():
    """Show legal corpus generation statistics"""
    print(f"Legal Corpus Generation Stats:")
    print(f"  Total corpus documents: {progress_manager.progress_data.get('total_qa_pairs_created', 0)}")
    print(f"  API requests: {progress_manager.progress_data.get('request_count', 0)}")

In [ ]:
# Cell 11: Utility Functions
def show_corpus_config():
    """Display current corpus configuration"""
    print("Corpus Synthesis Configuration:")
    print(f"  Domain: {DOMAIN_CONFIG['domain']}")
    print(f"  Approach: {DOMAIN_CONFIG['approach']}")
    print(f"  Source Dataset: {CONFIG['source_dataset']}")
    print(f"  Corpus Column: {CONFIG['corpus_column']}")
    print(f"  Variants per text: {DOMAIN_CONFIG['num_variants']}")
    print(f"  Max text length: {DOMAIN_CONFIG['max_text_length']}")

def update_corpus_config(corpus_column=None, approach=None, num_variants=None, max_text_length=None):
    """Update corpus configuration"""
    if corpus_column:
        CONFIG['corpus_column'] = corpus_column
    if approach:
        DOMAIN_CONFIG['approach'] = approach
    if num_variants:
        DOMAIN_CONFIG['num_variants'] = num_variants
    if max_text_length:
        DOMAIN_CONFIG['max_text_length'] = max_text_length
    
    print("Updated corpus configuration")
    show_corpus_config()

def show_progress():
    """Show current progress with enhanced file validation"""
    stats = progress_manager.progress_data
    print(f"Corpus Synthesis Progress:")
    print(f"  Current row: {stats.get('current_row', 0)}")
    print(f"  Processed rows: {len(stats.get('processed_rows', []))}")
    print(f"  QA pairs created: {stats['total_qa_pairs_created']}")
    print(f"  API requests: {stats['request_count']}")
    
    # Validate HF dataset compatibility
    validate_dataset_files()

# Keep the rest of Cell 11 as is...

## RUN

In [ ]:
# Initialize progress manager
progress_manager = ProgressManager()

print("Legal Corpus Generation Pipeline Ready!")
print("=" * 50)
print("Key Functions:")
print("- continue_legal_corpus_generation(target_count) - Generate legal documents")
print("- show_legal_corpus_stats() - Check statistics")
print("- show_progress() - Check current progress")

# Generate legal corpus
continue_legal_corpus_generation(target_count=1000000)

## END